# Temporal v3 Cached Batch Inspection

Load one batch from the daily-index rolling cache, convert it to the exact v3 torch batch, and inspect shapes, masks, labels, and a forward/loss pass.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(parent for parent in Path.cwd().resolve().parents if (parent / 'research').exists()) if Path.cwd().name != 'quant-research-workbench' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from research.mlops.rolling_loader.daily_index_dataset import AsyncDailyIndexBatchLoader
from research.temporal_event_model.v3.config import LoaderConfig, ModelConfig
from research.temporal_event_model.v3.data import batch_to_torch, loader_config_from_v3
from research.temporal_event_model.v3.losses import compute_loss
from research.temporal_event_model.v3.model import TemporalEventModelV3

CACHE_ROOT = Path('D:/market-data/prepared/daily_index_streaming_cache/events_daily_index_2019-02')
MONTHS = ('2019-02',)
BATCH_SIZE = 16
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_config = ModelConfig(d_model=128, event_layers=1, event_heads=4, fusion_layers=1, fusion_heads=4)
loader_config = LoaderConfig(cache_root=CACHE_ROOT, months=MONTHS, batch_size=BATCH_SIZE, read_workers=2, materialize_workers=2, loaded_parts_per_group=2, max_origins_per_epoch=BATCH_SIZE)
loader = AsyncDailyIndexBatchLoader(loader_config_from_v3(loader_config))
raw_batch = next(loader.iter_batches())
batch = batch_to_torch(raw_batch, model_config=model_config, device=device)
model = TemporalEventModelV3(model_config).to(device)
output = model(batch.x)
loss = compute_loss(output, batch)
print('device:', device)
print('samples:', batch.sample_count)
print('loss:', float(loss.loss.detach().cpu()))
print('active_task_count:', loss.metrics.get('train/active_task_count'))

In [ ]:
def show_tree(name, value, indent=0):
    pad = '  ' * indent
    if torch.is_tensor(value):
        print(f'{pad}{name}: tensor shape={tuple(value.shape)} dtype={value.dtype} device={value.device}')
    elif isinstance(value, dict):
        print(f'{pad}{name}: dict keys={list(value)[:12]}')
        for key in sorted(value):
            show_tree(str(key), value[key], indent + 1)
    else:
        print(f'{pad}{name}: {type(value).__name__}')

show_tree('x', batch.x)
show_tree('y', batch.y)
show_tree('output.future_bar_values', output.future_bar_values)
show_tree('output.intraday_logits', output.intraday_logits)
show_tree('output.corporate_action_logits', output.corporate_action_logits)

In [ ]:
print('identity preview')
for key, value in batch.identity.items():
    print(key, value[:5] if hasattr(value, '__getitem__') else value)

print('\nloader profile')
for key in sorted(batch.profile):
    print(f'{key}: {batch.profile[key]}')

print('\navailability')
for key, value in batch.x.get('input_availability', {}).items():
    if torch.is_tensor(value):
        print(key, float(value.float().mean().detach().cpu()))

print('\nloss metrics')
for key in sorted(loss.metrics):
    print(key, loss.metrics[key])